# BR-101 EDA Notebook 4: Weekly Local RNN Forecasting

This notebook reuses notebook 3's dense weekly panel, assigns train-derived `RGI` activity groups for analysis, and trains one recurrent model per activity group instead of one model per `RGI`.

Models in scope:

- per-activity-group `GRU`
- per-activity-group `LSTM`

Operational forecasting setup:

- train: `2017-W01` through `2023-W52`
- validation: `2024-W01` through `2024-W52`
- test: `2025-W01` through `2025-W52`
- lookback: `52` weeks
- latency gap: `4` weeks
- forecast horizon: `8` weeks
- forecast-origin stride: `4` weeks

Artifacts are written under `data/silver/notebook4_local_rnn_accident_count` and `data/gold/notebook4_local_rnn_accident_count`.


In [ ]:
from pathlib import Path
from time import perf_counter
import importlib

import pandas as pd
import seaborn as sns
from IPython.display import display

import sys
sys.path.insert(0, str(Path().resolve().parent))

from src.etl.silver import notebook4_pipeline as nb4
nb4 = importlib.reload(nb4)

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

CONFIG = nb4.Notebook4Config.from_project_root()
CONFIG.ensure_output_dirs()

print(f'Notebook 3 silver dir: {CONFIG.notebook3_silver_dir}')
print(f'Notebook 3 gold dir: {CONFIG.notebook3_gold_dir}')
print(f'Notebook 4 silver dir: {CONFIG.silver_output_dir}')
print(f'Notebook 4 gold dir: {CONFIG.gold_output_dir}')
print(nb4.tensorflow_status_message())


In [ ]:
inputs = nb4.load_notebook4_inputs(CONFIG)
weekly_rgi_panel = inputs['weekly_rgi_panel']
notebook3_benchmark_metrics = inputs['notebook3_benchmark_metrics']

weekly_rgi_panel.to_parquet(CONFIG.silver_output_dir / 'weekly_rgi_panel_reused.parquet', index=False)

print(f'Notebook 3 weekly panel rows: {len(weekly_rgi_panel):,}')
print(f'Retained RGIs: {weekly_rgi_panel["rgi_id"].nunique():,}')
display(weekly_rgi_panel.head())
display(weekly_rgi_panel['split'].value_counts(dropna=False).to_frame('rows'))

if not notebook3_benchmark_metrics.empty:
    display(notebook3_benchmark_metrics.sort_values(['split', 'rmse']).head(10))


In [ ]:
rgi_activity_summary = nb4.build_rgi_activity_summary(weekly_rgi_panel)
rgi_activity_summary = nb4.assign_activity_groups(rgi_activity_summary, CONFIG)
weekly_rgi_panel_with_groups = nb4.attach_activity_groups(weekly_rgi_panel, rgi_activity_summary)

rgi_activity_summary.to_parquet(CONFIG.silver_output_dir / 'rgi_activity_summary.parquet', index=False)
rgi_activity_summary.to_parquet(CONFIG.silver_output_dir / 'rgi_group_assignments.parquet', index=False)
weekly_rgi_panel_with_groups.to_parquet(CONFIG.silver_output_dir / 'weekly_rgi_panel_with_groups.parquet', index=False)

display(rgi_activity_summary.head(20))
display(rgi_activity_summary['group_name'].value_counts().rename_axis('group_name').to_frame('n_rgis'))


In [ ]:
group_diagnostics = nb4.build_group_diagnostics(weekly_rgi_panel_with_groups, rgi_activity_summary)
group_diagnostics.to_parquet(CONFIG.silver_output_dir / 'group_diagnostics.parquet', index=False)

display(group_diagnostics)
nb4.plot_group_diagnostics(weekly_rgi_panel_with_groups, rgi_activity_summary)


In [ ]:
local_datasets = nb4.prepare_local_rnn_datasets(weekly_rgi_panel_with_groups, CONFIG)
group_datasets = nb4.prepare_group_rnn_datasets(local_datasets, CONFIG)

sequence_manifest = nb4.build_sequence_manifest(local_datasets)
group_sequence_manifest = nb4.build_group_sequence_manifest(group_datasets)

sequence_manifest.to_parquet(CONFIG.silver_output_dir / 'sequence_manifest.parquet', index=False)
group_sequence_manifest.to_parquet(CONFIG.silver_output_dir / 'group_sequence_manifest.parquet', index=False)

display(sequence_manifest.head(20))
display(group_sequence_manifest)

print('RGIs with train samples:', int(sequence_manifest.loc[sequence_manifest['split'].eq('train'), 'n_samples'].gt(0).sum()))
print('RGIs with validation samples:', int(sequence_manifest.loc[sequence_manifest['split'].eq('validation'), 'n_samples'].gt(0).sum()))
print('RGIs with test samples:', int(sequence_manifest.loc[sequence_manifest['split'].eq('test'), 'n_samples'].gt(0).sum()))
print('Activity groups with train samples:', int(group_sequence_manifest.loc[group_sequence_manifest['split'].eq('train'), 'n_samples'].gt(0).sum()))
print('Activity groups with validation samples:', int(group_sequence_manifest.loc[group_sequence_manifest['split'].eq('validation'), 'n_samples'].gt(0).sum()))
print('Activity groups with test samples:', int(group_sequence_manifest.loc[group_sequence_manifest['split'].eq('test'), 'n_samples'].gt(0).sum()))


In [ ]:
nb4.require_tensorflow()

training_runs = []
training_timer = perf_counter()
for group_name, dataset in group_datasets.items():
    for architecture in ['gru', 'lstm']:
        print(f'Training {architecture.upper()} for activity group {group_name}...')
        training_runs.append(
            nb4.train_group_model(
                dataset['train'],
                dataset['validation'],
                CONFIG,
                architecture=architecture,
            )
        )

training_histories = nb4.flatten_training_histories(training_runs)
model_run_summary = nb4.build_model_run_summary(training_runs)
training_histories.to_parquet(CONFIG.gold_output_dir / 'training_histories.parquet', index=False)
model_run_summary.to_parquet(CONFIG.gold_output_dir / 'model_run_summary.parquet', index=False)

elapsed = perf_counter() - training_timer
print(f'Total training runtime: {elapsed:,.1f} seconds')
display(model_run_summary)
display(training_histories.tail(20))


In [ ]:
validation_frames = []
test_frames = []
for run in training_runs:
    dataset = group_datasets[run['group_name']]
    validation_frames.append(
        nb4.generate_rnn_forecasts(
            run['model'],
            dataset['validation'],
            architecture=run['architecture'],
        )
    )
    test_frames.append(
        nb4.generate_rnn_forecasts(
            run['model'],
            dataset['test'],
            architecture=run['architecture'],
        )
    )

validation_forecasts = nb4.combine_forecasts(*validation_frames)
test_forecasts = nb4.combine_forecasts(*test_frames)
combined_forecasts = nb4.combine_forecasts(validation_forecasts, test_forecasts)

for architecture in ['gru', 'lstm']:
    validation_forecasts.loc[validation_forecasts['architecture'].eq(architecture)].to_parquet(
        CONFIG.gold_output_dir / f'{architecture}_validation_forecasts.parquet',
        index=False,
    )
    test_forecasts.loc[test_forecasts['architecture'].eq(architecture)].to_parquet(
        CONFIG.gold_output_dir / f'{architecture}_test_forecasts.parquet',
        index=False,
    )

display(validation_forecasts.head())
display(test_forecasts.head())


In [ ]:
benchmark_metrics, group_level_metrics, rgi_level_metrics, horizon_level_metrics = nb4.evaluate_forecasts(
    weekly_rgi_panel_with_groups,
    combined_forecasts,
    CONFIG,
)
rgi_metric_report = nb4.build_rgi_metric_report(rgi_level_metrics)

benchmark_metrics.to_parquet(CONFIG.gold_output_dir / 'benchmark_metrics.parquet', index=False)
benchmark_metrics.to_csv(CONFIG.gold_output_dir / 'benchmark_metrics.csv', index=False)
group_level_metrics.to_parquet(CONFIG.gold_output_dir / 'group_level_metrics.parquet', index=False)
rgi_level_metrics.to_parquet(CONFIG.gold_output_dir / 'rgi_level_metrics.parquet', index=False)
rgi_metric_report.to_parquet(CONFIG.gold_output_dir / 'rgi_metric_report.parquet', index=False)
rgi_metric_report.to_csv(CONFIG.gold_output_dir / 'rgi_metric_report.csv', index=False)
horizon_level_metrics.to_parquet(CONFIG.gold_output_dir / 'horizon_level_metrics.parquet', index=False)

display(benchmark_metrics.sort_values(['split', 'rmse']))
display(group_level_metrics.sort_values(['split', 'group_name', 'rmse']))
display(horizon_level_metrics.sort_values(['split', 'group_name', 'horizon_step']))
display(rgi_metric_report.head(20))

if not notebook3_benchmark_metrics.empty:
    notebook3_rnn = notebook3_benchmark_metrics.loc[
        notebook3_benchmark_metrics['model'].astype(str).eq('rnn_global')
    ].copy()
    if not notebook3_rnn.empty:
        notebook3_rnn = notebook3_rnn.assign(comparison_source='notebook3_global_rnn')[
            ['comparison_source', 'model', 'split', 'rmse', 'forecast_row_coverage', 'n_predictions']
        ]
        notebook4_summary = benchmark_metrics.assign(comparison_source='notebook4_activity_group_rnn')[
            ['comparison_source', 'model', 'split', 'rmse', 'forecast_row_coverage', 'n_predictions']
        ]
        display(pd.concat([notebook3_rnn, notebook4_summary], ignore_index=True).sort_values(['split', 'rmse']))


In [ ]:
high_activity_rgis = rgi_activity_summary.loc[
    rgi_activity_summary['group_name'].eq('high_activity'), 'rgi_id'
].astype(str).tolist()
example_rgi = (
    high_activity_rgis[min(6, len(high_activity_rgis) - 1)]
    if high_activity_rgis
    else str(rgi_activity_summary['rgi_id'].iloc[0])
)

rgi_test_horizon_rmse = nb4.plot_test_predictions_for_rgi(
    weekly_rgi_panel_with_groups,
    test_forecasts,
    rgi_id=example_rgi,
    split='test',
)
display(rgi_test_horizon_rmse)

print('Per-RGI RMSE and R2 for every activity-group model (GRU and LSTM):')
display(rgi_metric_report)


## Closing Notes

After execution, summarize here:

- whether the activity-group models improved over notebook 3's global RNN
- which architecture won overall on validation and test
- which architecture won within each activity group
- how the per-RGI RMSE/R2 report differs across low-, mid-, and high-activity RGIs
- what should change next if another modeling notebook is needed
